# Final Testing and Model Comparison

Evaluates **Model A (Custom CNN)** and **Model B (MobileNetV2)** on the test split, which no earlier notebook has used.

- Same test images and same preprocessing for both models.
- Each model's decision threshold was chosen on **validation** and is only **read** here. Nothing is tuned on test.
- Both models are reported at threshold 0.50 and at their validation threshold.

## 1. Setup

In [ ]:
!pip -q install datasets huggingface_hub

In [ ]:
import os, json, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras

if not hasattr(keras, "KerasTensor"):
    from keras.src.backend import KerasTensor
    keras.KerasTensor = KerasTensor

from pathlib import Path
from PIL import Image as PILImage
from datasets import load_dataset
from tensorflow.keras import layers
from tqdm.auto import tqdm
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                             classification_report, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)

MODELS = {"Custom CNN": "custom_cnn", "MobileNetV2": "mobilenetv2"}   # name -> file prefix
SHORT = {"Custom CNN": "cnn", "MobileNetV2": "mnv2"}
CLASS_NAMES = ["Real", "AI-Generated"]
IMAGE_SIZE = 224
IMAGE_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, 3)
BATCH_SIZE = 64
SEED = 42

DATASET_ID = "Rajarshi-Roy-research/Defactify_Image_Dataset"
REPO_URL = "https://github.com/Michaelatj/AI-Generated-Image-Classification-using-CNN.git"
GIT_USER_NAME = "Michael Andreas Tjendra"
GIT_USER_EMAIL = "alongbest2000@gmail.com"
GITHUB_USERNAME = "Michaelatj"

IN_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IN_KAGGLE else Path("/content")
HF_CACHE_DIR = "/kaggle/temp/huggingface_cache" if IN_KAGGLE else "/content/huggingface_cache"
REPO_DIR = WORK_DIR / "AI-Generated-Image-Classification-using-CNN"

tf.keras.utils.set_random_seed(SEED)
print("Environment:", "Kaggle" if IN_KAGGLE else "Colab/other", "| TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if subprocess.check_output(["git", "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Repository contains local changes. Check git status before syncing.")
subprocess.run(["git", "pull", "--rebase", "origin", "main"], check=True)

ARTIFACT_DIR = REPO_DIR / "artifacts"
SPLIT_DIR = ARTIFACT_DIR / "splits"
CONFIG_DIR = ARTIFACT_DIR / "configs"
MODEL_DIR = ARTIFACT_DIR / "models"
EVAL_DIR = ARTIFACT_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load models and test data

In [ ]:
@keras.saving.register_keras_serializable(package="aigen")
class MobileNetV2Preprocess(layers.Layer):
    """[0, 255] -> [-1, 1]; same definition as in training."""
    def call(self, x):
        return tf.cast(x, tf.float32) / 127.5 - 1.0


cfgs, nets, THRESH = {}, {}, {}
for name, prefix in MODELS.items():
    stem = f"{prefix}"
    cfgs[name] = json.load(open(MODEL_DIR / f"{stem}_config.json", encoding="utf-8"))
    nets[name] = tf.keras.models.load_model(MODEL_DIR / f"{stem}_best.keras")
    THRESH[name] = float(cfgs[name]["decision_threshold"])
    print(f"{name:12s} params={nets[name].count_params():,}  size={cfgs[name]['model_size_mb']} MB  "
          f"threshold (from validation)={THRESH[name]:.2f}")

# both models must come from the same data and preprocessing
assert cfgs["Custom CNN"]["dataset_revision"] == cfgs["MobileNetV2"]["dataset_revision"]
assert (cfgs["Custom CNN"]["preprocessing"]["center_square_crop"]
        == cfgs["MobileNetV2"]["preprocessing"]["center_square_crop"])
HF_REVISION = cfgs["Custom CNN"]["dataset_revision"]
CENTER_CROP = cfgs["Custom CNN"]["preprocessing"]["center_square_crop"]

In [ ]:
sort_cols = ["source_split", "row_idx"]
train_df = pd.read_csv(SPLIT_DIR / "train.csv")
val_df = pd.read_csv(SPLIT_DIR / "validation.csv")
test_df = pd.read_csv(SPLIT_DIR / "test.csv").sort_values(sort_cols).reset_index(drop=True)
print("Test:", test_df.shape, "| class share:", test_df["label_a"].value_counts(normalize=True).round(3).to_dict())

# no caption group of the test split may appear in train or validation
for name, other in [("train", train_df), ("validation", val_df)]:
    assert not (set(test_df["caption_group_id"]) & set(other["caption_group_id"])), f"LEAKAGE with {name}"
print("No caption-group overlap between test and train/validation.")

HF_DATASETS = {s: load_dataset(DATASET_ID, split=s, revision=HF_REVISION, cache_dir=HF_CACHE_DIR)
               for s in sorted(set(test_df["source_split"]))}


def center_square(image):
    w, h = image.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    return image.crop((left, top, left + s, top + s))


def prepare_image(image):
    """Same as the evaluation branch used in training (and later in the app)."""
    image = image.convert("RGB")
    if CENTER_CROP:
        image = center_square(image)
    return image.resize((IMAGE_SIZE, IMAGE_SIZE), resample=PILImage.Resampling.BILINEAR)


def test_generator():
    for row in test_df.itertuples(index=False):
        image = HF_DATASETS[row.source_split][int(row.row_idx)]["Image"]
        yield np.asarray(prepare_image(image), dtype=np.uint8)

## 3. Test predictions
Images are decoded once and fed to both models. Probabilities are cached, so a re-run does not repeat the ~10 minute pass.

In [ ]:
PROBS_PATH = EVAL_DIR / f"final_test_probs.csv"

if PROBS_PATH.exists():
    probs_df = pd.read_csv(PROBS_PATH)
    assert list(probs_df["image_id"]) == list(test_df["image_id"])
    print("Loaded cached probabilities.")
else:
    ds = (tf.data.Dataset.from_generator(test_generator,
          output_signature=tf.TensorSpec(shape=IMAGE_SHAPE, dtype=tf.uint8))
          .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
    out = {n: [] for n in MODELS}
    for images in tqdm(ds, total=int(np.ceil(len(test_df) / BATCH_SIZE)), desc="Predicting"):
        for name, net in nets.items():
            out[name].append(np.asarray(net.predict_on_batch(images)).ravel())
    probs_df = pd.DataFrame({"image_id": test_df["image_id"]})
    for name in MODELS:
        probs_df[f"p_{SHORT[name]}"] = np.concatenate(out[name])
    probs_df.to_csv(PROBS_PATH, index=False)

res = test_df[["image_id", "source_split", "row_idx", "label_a", "label_b_name"]].copy()
for name in MODELS:
    res[f"p_{SHORT[name]}"] = probs_df[f"p_{SHORT[name]}"].to_numpy()
y_true = res["label_a"].to_numpy().astype(int)

## 4. Test results
Four scenarios: each model at threshold 0.50 and at its validation threshold.

In [ ]:
# Calculate classification metrics at a given threshold
def metrics_at(y, p, t):
    pred = (p >= t).astype(int)

    return {
        "threshold": round(float(t), 2),
        "accuracy": accuracy_score(y, pred),
        "precision_ai": precision_score(y, pred, zero_division=0),
        "recall_ai": recall_score(y, pred, zero_division=0),
        "recall_real": recall_score(y, pred, pos_label=0, zero_division=0),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
    }

In [ ]:
# Compare model performance using default and validation thresholds
rows = []

for name in MODELS:
    p = res[f"p_{SHORT[name]}"].to_numpy()

    for label, t in [
        ("threshold 0.50", 0.5),
        ("validation threshold", THRESH[name])
    ]:
        rows.append({
            "model": name,
            "scenario": label,
            **metrics_at(y_true, p, t),
            "roc_auc": roc_auc_score(y_true, p)
        })

scenarios = pd.DataFrame(rows)

display(scenarios.round(4))

In [ ]:
# Display detailed classification metrics for each model
for name in MODELS:
    p = res[f"p_{SHORT[name]}"].to_numpy()

    print(f"\n{name} (Validation Threshold: {THRESH[name]:.2f})")

    display(
        pd.DataFrame(
            classification_report(
                y_true,
                (p >= THRESH[name]).astype(int),
                target_names=CLASS_NAMES,
                output_dict=True,
                zero_division=0
            )
        ).transpose().round(4)
    )

In [ ]:
# CONFUSION MATRIX - Custom CNN

# Mengambil seluruh nama model dari dictionary MODELS
MODEL_NAMES = list(MODELS.keys())

name = MODEL_NAMES[0]

# Mengambil probabilitas prediksi dari hasil evaluasi
p = res[f"p_{SHORT[name]}"].to_numpy()

# Mengubah probabilitas menjadi prediksi kelas berdasarkan threshold
y_pred = (p >= THRESH[name]).astype(int)

# Menghitung confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

# Membuat plot confusion matrix
fig, ax = plt.subplots(figsize=(5, 4.5))

ax.imshow(cm, cmap="Blues")

# Mengatur label sumbu X dan Y
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticklabels(CLASS_NAMES)

ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"{name} (Test, Threshold {THRESH[name]:.2f})")

# Menampilkan jumlah data pada setiap sel confusion matrix
for i in range(2):
    for j in range(2):
        ax.text(
            j, i,
            f"{cm[i, j]:,}",
            ha="center",
            va="center",
            color="white" if cm[i, j] > cm.max() / 2 else "black"
        )

plt.tight_layout()

# Menyimpan hasil confusion matrix Model 1
plt.savefig(
    EVAL_DIR / f"final_confusion_{SHORT[name]}.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# CONFUSION MATRIX - MobileNetV2
name = MODEL_NAMES[1]

# Mengambil probabilitas prediksi dari hasil evaluasi
p = res[f"p_{SHORT[name]}"].to_numpy()

# Mengubah probabilitas menjadi prediksi kelas berdasarkan threshold
y_pred = (p >= THRESH[name]).astype(int)

# Menghitung confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

# Membuat plot confusion matrix
fig, ax = plt.subplots(figsize=(5, 4.5))

ax.imshow(cm, cmap="Blues")

# Mengatur label sumbu X dan Y
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticklabels(CLASS_NAMES)

ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"{name} (Test, Threshold {THRESH[name]:.2f})")

# Menampilkan jumlah data pada setiap sel confusion matrix
for i in range(2):
    for j in range(2):
        ax.text(
            j, i,
            f"{cm[i, j]:,}",
            ha="center",
            va="center",
            color="white" if cm[i, j] > cm.max() / 2 else "black"
        )

plt.tight_layout()

# Menyimpan hasil confusion matrix Model 2
plt.savefig(
    EVAL_DIR / f"final_confusion_{SHORT[name]}.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ROC CURVE COMPARISON
fig, ax = plt.subplots(figsize=(6, 5))

for name in MODELS:
    p = res[f"p_{SHORT[name]}"].to_numpy()
    fpr, tpr, _ = roc_curve(y_true, p)
    auc_score = roc_auc_score(y_true, p)
    ax.plot(
        fpr,
        tpr,
        label=f"{name} (AUC={auc_score:.4f})"
    )

# Menambahkan garis baseline sebagai pembanding
ax.plot(
    [0, 1],
    [0, 1],
    "--",
    color="gray",
    label="Random Classifier"
)

# Mengatur label dan judul plot
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison (Test)")

# Menampilkan legend dan grid
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()

# Menyimpan hasil ROC Curve
plt.savefig(
    EVAL_DIR / f"final_roc_comparison.png",
    dpi=150,
    bbox_inches="tight"
)

plt.show()

## 5. Accuracy per generator

In [ ]:
rows = []
for name in MODELS:
    pred = (res[f"p_{SHORT[name]}"] >= THRESH[name]).astype(int)
    for gname, g in res.assign(pred=pred).groupby("label_b_name"):
        rows.append({"model": name, "source": gname, "images": len(g),
                     "accuracy": float((g["pred"] == g["label_a"]).mean())})
per_gen = pd.DataFrame(rows)
wide = per_gen.pivot(index="source", columns="model", values="accuracy")
display(wide.round(4))

In [ ]:
ax = wide.plot(kind="bar", figsize=(10, 4.5))
ax.set_ylim(0.5, 1.02); ax.set_ylabel("Accuracy (test)"); ax.set_xlabel("")
ax.set_title("Accuracy per source (validation thresholds)"); ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(EVAL_DIR / f"final_per_generator.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Error analysis
The most confident mistakes of the better model (by test macro-F1): Real images it called AI (false positives) and AI images it called Real (false negatives).

In [ ]:
# Select the best model based on validation Macro F1
best_name = (
    scenarios[scenarios["scenario"] == "validation threshold"]
    .sort_values("macro_f1", ascending=False)
    .iloc[0]["model"]
)

pc = f"p_{SHORT[best_name]}"

print("Selected model:", best_name)

In [ ]:
# Function to visualize misclassified images
def show_cases(rows, title, fname):
    fig, axes = plt.subplots(2, 4, figsize=(13.5, 8.2))

    for ax in axes.ravel():
        ax.axis("off")

    for ax, (_, r) in zip(axes.ravel(), rows.iterrows()):
        image = HF_DATASETS[r["source_split"]][int(r["row_idx"])]["Image"]
        ax.imshow(prepare_image(image))
        ax.set_title(f"{r['label_b_name']}\nP(AI)={r[pc]:.2f}", fontsize=9)

    fig.suptitle(title)
    plt.tight_layout(h_pad=2.0)
    plt.savefig(EVAL_DIR / fname, dpi=130, bbox_inches="tight")
    plt.show()

In [ ]:
# Visualize real images incorrectly classified as AI
fp_cases = res[res["label_a"] == 0].nlargest(8, pc)

show_cases(
    fp_cases,
    f"{best_name}: Real Images Called AI (False Positives)",
    f"final_errors_fp.png"
)

In [ ]:
# Visualize AI images incorrectly classified as real
fn_cases = res[res["label_a"] == 1].nsmallest(8, pc)

show_cases(
    fn_cases,
    f"{best_name}: AI Images Called Real (False Negatives)",
    f"final_errors_fn.png"
)

## 7. Unique feature 1: decision threshold chosen on validation
**What:** the threshold is not fixed at 0.5. It is chosen on the validation set (max macro-F1) and stored with each model.

**Why:** the data is 5:1 imbalanced and class weights shift the probabilities, so 0.5 is not the natural cut-off.

**Impact on test:** change from threshold 0.50 to the validation threshold.

In [ ]:
cmp_cols = ["accuracy", "macro_f1", "balanced_accuracy", "recall_real", "recall_ai"]
rows = []
for name in MODELS:
    a = scenarios[(scenarios["model"] == name) & (scenarios["scenario"] == "threshold 0.50")].iloc[0]
    b = scenarios[(scenarios["model"] == name) & (scenarios["scenario"] == "validation threshold")].iloc[0]
    rows.append({"model": name, "threshold": b["threshold"],
                 **{f"{c} (0.50 -> tuned)": f"{a[c]:.4f} -> {b[c]:.4f}" for c in cmp_cols}})
display(pd.DataFrame(rows).set_index("model").T)

## 8. Unique feature 2: Grad-CAM
**What:** a heatmap of the image regions that raised the model's P(AI-Generated).

**Why:** a detector should show what it looked at, and heatmaps can reveal shortcuts (for example, attending to borders instead of content).

**How:** the gradient of P(AI) with respect to the last convolutional feature map (the input of the `gap` layer in both models) is averaged per channel and used to weight that map; only positive evidence is kept.

A heatmap explains the model's behaviour. It does not prove the model is right.

In [ ]:
def make_gradcam_model(model):
    return keras.Model(model.inputs, [model.get_layer("gap").input, model.output])

grad_models = {n: make_gradcam_model(net) for n, net in nets.items()}

def gradcam(grad_model, images_uint8):
    x = tf.convert_to_tensor(images_uint8, dtype=tf.float32)
    with tf.GradientTape() as tape:
        tape.watch(x)
        conv, pred = grad_model(x, training=False)
        score = pred[:, 0]
    grads = tape.gradient(score, conv)
    weights = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.nn.relu(tf.reduce_sum(conv * weights[:, None, None, :], axis=-1))
    cam = cam / (tf.reduce_max(cam, axis=(1, 2), keepdims=True) + 1e-8)
    return cam.numpy(), pred.numpy().ravel()

# 4 examples chosen by the better model: one correct AI, one correct Real, one false positive, one false negative
pred_best = (res[pc] >= THRESH[best_name]).astype(int)
picks = [
    ("AI, correct", res[(res["label_a"] == 1) & (pred_best == 1)].sample(1, random_state=SEED)),
    ("Real, correct", res[(res["label_a"] == 0) & (pred_best == 0)].sample(1, random_state=SEED)),
    ("Real called AI", res[res["label_a"] == 0].nlargest(1, pc)),
    ("AI called Real", res[res["label_a"] == 1].nsmallest(1, pc)),
]
case_df = pd.concat([d.assign(category=c) for c, d in picks], ignore_index=True)
batch = np.stack([np.asarray(prepare_image(HF_DATASETS[r["source_split"]][int(r["row_idx"])]["Image"]))
                  for _, r in case_df.iterrows()])
cams = {n: gradcam(gm, batch) for n, gm in grad_models.items()}

fig, axes = plt.subplots(len(case_df), 3, figsize=(10, 3.3 * len(case_df)))
for i, (_, r) in enumerate(case_df.iterrows()):
    axes[i, 0].imshow(batch[i])
    axes[i, 0].set_title(f"{r['category']}\n{r['label_b_name']}", fontsize=9)
    for j, name in enumerate(MODELS, start=1):
        cam, prob = cams[name][0][i], cams[name][1][i]
        axes[i, j].imshow(batch[i])
        axes[i, j].imshow(cam, cmap="jet", alpha=0.45, extent=(0, IMAGE_SIZE, IMAGE_SIZE, 0), interpolation="bilinear")
        axes[i, j].set_title(f"{name}: P(AI)={prob:.2f} -> {'AI' if prob >= THRESH[name] else 'Real'}", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.savefig(EVAL_DIR / f"final_gradcam.png", dpi=140, bbox_inches="tight")
plt.show()

## 9. Save results and push
`deployment_config_v3.json` tells the app which model files to load, with which threshold and preprocessing.

In [ ]:
scenarios.to_csv(EVAL_DIR / f"final_test_scenarios.csv", index=False)
per_gen.to_csv(EVAL_DIR / f"final_per_generator.csv", index=False)

deployment_config = {
    "class_names": CLASS_NAMES,
    "default_model": best_name,
    "preprocessing": {"image_size": IMAGE_SIZE, "color_mode": "RGB", "center_square_crop": CENTER_CROP,
                      "resize_interpolation": "bilinear",
                      "model_input": "raw 0-255 RGB, 224x224x3 (scaling is inside each model)"},
    "models": {name: {"file": f"models/{cfgs[name]['model_file']}", "threshold": THRESH[name],
                      "needs_custom_layer": prefix == "mobilenetv2"}
               for name, prefix in MODELS.items()},
}
DEPLOY_PATH = CONFIG_DIR / f"deployment_config.json"
json.dump(deployment_config, open(DEPLOY_PATH, "w", encoding="utf-8"), indent=4)
print("Saved", DEPLOY_PATH.name, "| default model:", best_name)

In [ ]:
def get_github_token():
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except ImportError:
        pass
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception as e:
        print("Colab secret unavailable:", type(e).__name__)
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        import getpass
        token = getpass.getpass("Paste GITHUB_TOKEN (hidden input): ")
    return token


def push_to_github(files, message):
    token = get_github_token()
    files = [Path(p) for p in files if Path(p).exists()]
    subprocess.run(["git", "config", "--local", "user.name", GIT_USER_NAME], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "config", "--local", "user.email", GIT_USER_EMAIL], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "add", "--"] + [str(f.relative_to(REPO_DIR)) for f in files], cwd=REPO_DIR, check=True)
    if subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=REPO_DIR).returncode != 0:
        subprocess.run(["git", "commit", "-m", message], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "fetch", "origin"], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "pull", "--rebase", "origin", "main"], cwd=REPO_DIR, check=True)

    askpass = Path("/tmp/github-askpass.sh")
    askpass.write_text("""#!/bin/sh
case "$1" in
    *sername*) printf '%s\\n' "$GITHUB_USERNAME" ;;
    *assword*) printf '%s\\n' "$GITHUB_TOKEN" ;;
    *) exit 1 ;;
esac
""")
    os.chmod(askpass, 0o700)
    env = os.environ.copy()
    env.update({"GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0",
                "GITHUB_TOKEN": token, "GITHUB_USERNAME": GITHUB_USERNAME})
    result = subprocess.run(["git", "push", "origin", "main"], cwd=REPO_DIR, env=env,
                            capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("GitHub push failed.")
    print("Pushed:", [f.name for f in files])


push_to_github(sorted(EVAL_DIR.glob(f"final")) + [DEPLOY_PATH],
               f"Add final test evaluation and deployment config")